## Why Do We Still Need Preference Learning If We Already Have SFT?

The key is not that SFT is incapable of handling these tasks, but rather that **for some tasks, it's easier to judge which of two responses is better than to write a perfect response from scratch.**

For example, suppose a user asks:

`Write a comforting message to a friend who failed an exam.`

We could generate 10 different responses. **All of them might be correct**, grammatically fluent, and aligned with the instruction. However, some might feel warmer, others might sound too preachy, and some might be unnecessarily verbose.

In this case, a human might struggle to write a single ideal response, but find it easy to choose the one they prefer when comparing two responses.


---

# Bradley–Terry Model (BT Model)

## 1. What Problem Does It Solve?

The **Bradley–Terry Model**, introduced in **1952**, is a probabilistic model for pairwise comparisons.

**Core question:** Given observations of who wins against whom, can we estimate each competitor's underlying strength and predict the probability of future wins?

The key idea is that **a stronger competitor is more likely to win, but is not guaranteed to win**.

## 2. The Basic Model

Suppose three AI models have the following hypothetical strength scores:

| Model | Strength |
|---|---|
| A | 8 |
| B | 4 |
| C | 2 |

The probability that A beats B is:

$$
P(A>B)=\frac{s_A}{s_A+s_B}
$$

For example:

$$
P(A>B)=\frac{8}{8+4}=66.7\%
$$

Other comparisons:

- A beats C: 80%
- B beats C: 66.7%
- A beats B: 66.7%

**Important:** These scores are hypothetical. In practice, the strengths are unknown and must be learned from observed comparisons.

## 3. Learning Strengths from Data

Suppose we observe:

| Comparison | Results |
|---|---|
| A vs. B | A wins 70 out of 100 |
| A vs. C | A wins 85 out of 100 |
| B vs. C | B wins 65 out of 100 |

We want to **reverse-engineer the hidden strengths** that best explain these outcomes.

The BT Model estimates these strengths by maximizing the likelihood of the observed comparison results.

Thus:

**Pairwise outcomes → Learn hidden strengths → Predict future win probabilities**

The estimated scores explain the comparisons collectively, rather than necessarily reproducing each observed win rate exactly.

## 4. Connection to Softmax and Sigmoid

Instead of using positive strength scores directly, we can represent strength using arbitrary real-valued scores:

$$
s_A=e^{r_A}, \quad s_B=e^{r_B}
$$

Substituting into the BT formula gives:

$$
P(A>B)=\frac{e^{r_A}}{e^{r_A}+e^{r_B}}
$$

This is exactly **Softmax with two options**.

We can simplify it further:

$$
P(A>B)=\frac{1}{1+e^{-(r_A-r_B)}}
$$

Therefore:

$$
\boxed{P(A>B)=\sigma(r_A-r_B)}
$$

where $\sigma$ is the sigmoid function.

**Key insight:** Only the difference between scores matters, not their absolute values.

- Equal scores → 50% probability
- A higher score for A → A is more likely to win
- A larger score difference → Stronger preference for the higher-scored option

## 5. Connection to LLM Training (RLHF)

The Bradley–Terry Model is commonly used in **Reward Modeling**, a component of traditional Reinforcement Learning from Human Feedback (RLHF).

### The Problem

Humans often find it easier to compare two answers than to assign each answer an absolute quality score.

For example:

**Prompt:** What is machine learning?

- **Answer A:** A technical definition involving statistical algorithms.
- **Answer B:** A simple explanation using examples.

A human evaluator prefers B.

### How It Works

1. Generate two answers to the same prompt.
2. Ask a human evaluator which answer is better.
3. Collect many pairwise preferences.
4. Train a Reward Model to assign scores to answers.
5. Use the Bradley–Terry probability model to predict human preferences.

For two responses, the model predicts:

$$
P(B>A)=\sigma(r_B-r_A)
$$

During training, the Reward Model adjusts its parameters so that preferred answers receive higher scores.

**Core insight:** Instead of asking humans to define an exact reward function, we can learn one from relative preferences.

## 6. Limitations

The classical Bradley–Terry Model assumes that each competitor has a single underlying strength that explains pairwise outcomes.

However, preferences are not always globally consistent.

### Example: Rock–Paper–Scissors

- Rock beats Scissors.
- Scissors beats Paper.
- Paper beats Rock.

No single global strength ranking can represent all three advantages simultaneously.

This illustrates a limitation of the model: **context-dependent or cyclic preferences cannot always be explained by one scalar strength per competitor**.

Similarly, human preferences for AI responses can depend on the question, evaluator, and context.

## 7. Key Takeaways

- **Bradley–Terry (1952)** is a classical probabilistic model, not a recent invention.
- It learns hidden strengths from pairwise comparison data.
- Higher strength implies a higher probability of winning, not guaranteed victory.
- Its log-strength formulation is mathematically equivalent to two-class Softmax or Sigmoid.
- It is useful in LLM Reward Modeling because humans can often compare answers more reliably than assign absolute scores.
- Its main limitation is the assumption that preferences can be represented by scalar scores.

**Mental Model**

> Bradley–Terry converts relative comparisons into a probabilistic scoring system.

**Connection to LLMs**

> Human preferences → Pairwise comparisons → Reward Model → Bradley–Terry / Sigmoid → Preference probabilities → Training signal

# Understanding the Bradley–Terry Formula

## 1. The Formula

$$
P(A\succ B)=\sigma(r_A-r_B)
$$

**In plain English:**

The probability that A is preferred over B depends on the difference between their reward scores.

## 2. Mathematical Notation

| Symbol | Meaning | Explanation |
|---|---|---|
| $P(\cdot)$ | Probability | The probability of an event |
| $\succ$ | Is preferred to | $A\succ B$ means A is preferred over B |
| $\sigma$ | Sigma | The Greek letter used to denote the sigmoid function |
| $r_A$ | Reward score of A | The model's score for A |
| $r_B$ | Reward score of B | The model's score for B |
| $r_A-r_B$ | Score difference | How much higher or lower A scores than B |

### Why use $\succ$ instead of $>$?

- $A>B$ usually represents a numerical comparison.
- $A\succ B$ represents a **preference relation**.

For example:

$$
\text{Apple}\succ\text{Banana}
$$

This means someone prefers apples to bananas, not that apples are numerically greater than bananas.

## 3. What Is the Sigmoid Function?

The sigmoid function is defined as:

$$
\sigma(x)=\frac{1}{1+e^{-x}}
$$

It converts any real number into a value between 0 and 1.

This makes it useful for modeling probabilities.

| Input $x$ | Sigmoid $\sigma(x)$ |
|---|---|
| $-3$ | 0.047 |
| $-1$ | 0.269 |
| $0$ | 0.500 |
| $1$ | 0.731 |
| $3$ | 0.953 |

**Key properties:**

- Negative input → Probability below 50%
- Zero input → Probability equals 50%
- Positive input → Probability above 50%
- Larger positive input → Probability approaches 100%
- Larger negative input → Probability approaches 0%

The output approaches but never reaches exactly 0 or 1 for finite inputs.

## 4. Numerical Example

Suppose a Reward Model assigns:

$$
r_A=3,\qquad r_B=2
$$

The score difference is:

$$
r_A-r_B=1
$$

Applying sigmoid:

$$
P(A\succ B)=\sigma(1)
$$

$$
=\frac{1}{1+e^{-1}}\approx0.731
$$

**Interpretation:** The model predicts a 73.1% probability that A will be preferred over B.

## 5. Why Use Sigmoid?

Reward scores can be any real numbers, including negative values.

However, probabilities must satisfy:

$$
0\le P\le1
$$

Simply using the score difference as a probability would not work.

For example, a score difference of 3 cannot directly represent a probability of 300%.

Sigmoid solves this by mapping any score difference into a valid probability.

It also preserves the ordering of scores: a larger difference produces a larger preference probability.

## 6. Important Properties

### Only the Score Difference Matters

These two pairs produce the same probability:

$$
(r_A,r_B)=(3,2)
$$

$$
(r_A,r_B)=(103,102)
$$

Because both have:

$$
r_A-r_B=1
$$

The absolute reward values do not matter; only their difference matters.

### Preference Probabilities Are Complementary

For a binary choice without ties:

$$
P(A\succ B)+P(B\succ A)=1
$$

This follows from the sigmoid identity:

$$
\sigma(x)+\sigma(-x)=1
$$

### Connection to Softmax

The Bradley–Terry formula can also be expressed as:

$$
P(A\succ B)=\frac{e^{r_A}}{e^{r_A}+e^{r_B}}
$$

Dividing numerator and denominator by $e^{r_A}$ gives:

$$
P(A\succ B)=\frac{1}{1+e^{r_B-r_A}}
$$

Therefore:

$$
\boxed{P(A\succ B)=\sigma(r_A-r_B)}
$$

**Two-class Softmax is mathematically equivalent to Sigmoid applied to the score difference.**

## 7. Mental Model

The entire formula represents a simple pipeline:

**Reward scores → Score difference → Sigmoid → Preference probability**

The notation looks complicated, but the underlying idea is simple:

> Compare two scores, measure their difference, and convert that difference into a probability.

**Key distinction:** The reward scores are learned estimates, and the probability is a model prediction—not a guarantee of what every human evaluator will choose.

## A Beautiful Mathematical Connection: Bradley–Terry and Softmax

Recall that:

$$
\sigma(r_A-r_B)=\frac{e^{r_A}}{e^{r_A}+e^{r_B}}
$$

**Isn't this exactly Softmax with two options?**

Suppose we reduce the vocabulary in LLM pretraining to just two tokens, A and B.

The mathematical form of the pretraining objective would then be identical to the **Bradley–Terry Reward Loss**!

However, the two models learn fundamentally different things:

- **Pretraining:** Given a context, which token comes next?
- **Reward Model:** Given two responses to the same prompt, which one would a human prefer?

The scores also represent different quantities:

- **Pretraining:** The model outputs a logit for each candidate token.
- **Reward Model:** The model typically outputs a single scalar reward for each complete prompt-response pair.

**Key insight:** Both training objectives follow the same fundamental pattern:

$$
\boxed{\text{Scores}\rightarrow\text{Probabilities}\rightarrow\text{Loss}\rightarrow\text{Gradient Descent}}
$$

The key difference is **what the model is predicting and what the training data defines as the correct answer**.